In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica (RandomForestRegressor) aprendida apenas com dados sem falha
+ Aplicação global em toda a base (sem e com falha)
+ Classificação de falhas (RandomForestClassifier) com split por temperatura
Base utilizada: base-completaa.pkl (3 classes: 0=sem, 1=falha1, 2=falha2)
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"   # ✅ base completa (3 classes)
REF_TEMP = 20
FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 59
SMOOTH_WIN = 5
PLOT_N_EXAMPLES = 4

RF_COMP_PARAMS = dict(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

RF_CLASSIF_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# ========= FUNÇÕES AUXILIARES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f / 1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def add_extra_features(X):
    mu  = X.mean(axis=1, keepdims=True)
    sd  = X.std(axis=1,  keepdims=True)
    amp = (X.max(axis=1)-X.min(axis=1)).reshape(-1,1)
    return np.hstack([X, mu, sd, amp])

def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec).reshape(-1,1).astype(float)])

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr
    r = win // 2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s / float(win)

# ========= ETAPA 1 – CARREGAMENTO =========
print("🔹 Carregando base completa...")
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz / 1e3

print(f"Faixa de frequência: {FREQ_MIN_KHZ}–{FREQ_MAX_KHZ} kHz")
print(f"Nº amostras: {len(df)} | Nº features: {len(fcols)}")

# ========= ETAPA 2 – DIVISÃO (sem falha / com falha) =========
df_sem = df[df["falha"] == 0].copy()
print(f"Amostras sem falha: {len(df_sem)} | total: {len(df)}")

# ========= ETAPA 3 – REFERÊNCIA REAL @20°C =========
pool_20 = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
if len(pool_20) == 0:
    print("⚠️ Nenhum dado sem falha exato a 20°C, usando mediana global sem falha.")
    y_ref = np.median(df_sem[fcols].to_numpy(float), axis=0)
else:
    y_ref = np.median(pool_20, axis=0)
print(f"Referência calculada com {len(pool_20)} curvas sem falha @ {REF_TEMP}°C.")

# ========= ETAPA 4 – TREINO RF-COMP (apenas sem falha) =========
X_sem = df_sem[fcols].to_numpy(float)
T_sem = df_sem["temperatura_c"].to_numpy(float)
Y_target = (y_ref[None, :] - X_sem)

X_aug_sem = add_extra_features(X_sem)
X_comp_sem = add_temp_feature(X_aug_sem, T_sem)

print("\n🔹 Treinando RandomForestRegressor (compensação térmica)...")
t0 = time.time()
rf_comp = RandomForestRegressor(**RF_COMP_PARAMS)
rf_comp.fit(X_comp_sem, Y_target)
print(f"✅ RF-Comp treinado em {time.time()-t0:.1f}s")

# ========= ETAPA 5 – APLICA COMPENSAÇÃO EM TODA A BASE =========
X_all = df[fcols].to_numpy(float)
T_all = df["temperatura_c"].to_numpy(float)
X_aug_all = add_extra_features(X_all)
X_comp_all = add_temp_feature(X_aug_all, T_all)

print("🔹 Aplicando compensação térmica em toda a base...")
Y_hat = X_all + rf_comp.predict(X_comp_all)
if SMOOTH_WIN > 1 and SMOOTH_WIN % 2 == 1:
    for i in range(Y_hat.shape[0]):
        Y_hat[i] = moving_average(Y_hat[i], SMOOTH_WIN)

df_comp = df.copy()
df_comp[fcols] = Y_hat
print("✅ Compensação aplicada com sucesso.")

# ========= ETAPA 6 – CLASSIFICAÇÃO DE FALHAS (split por temperatura) =========
print("\n🔹 Dividindo treino/teste por temperatura...")

# Divide as temperaturas (70% treino, 30% teste)
temps_all = sorted(df_comp["temperatura_c"].unique())
n_train = int(len(temps_all) * 0.7)
np.random.seed(42)
temps_train = set(np.random.choice(temps_all, n_train, replace=False))
temps_test = set(t for t in temps_all if t not in temps_train)

print(f"Temperaturas treino: {sorted(temps_train)}")
print(f"Temperaturas teste:  {sorted(temps_test)}")

# Cria subconjuntos
df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)].copy()
df_test  = df_comp[df_comp["temperatura_c"].isin(temps_test)].copy()

X_train = df_train[fcols].to_numpy(float)
y_train = df_train["falha"].to_numpy(int)
X_test  = df_test[fcols].to_numpy(float)
y_test  = df_test["falha"].to_numpy(int)

print(f"Amostras treino: {len(X_train)} | teste: {len(X_test)}")

# Treina classificador multiclasse
print("\n🔹 Treinando RandomForestClassifier (multiclasse)...")
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n== RESULTADOS RANDOM FOREST (após compensação, 3 classes, split por temperatura) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

# ========= ETAPA 7 – PLOT DE EXEMPLOS =========
def pick_examples(df, n=PLOT_N_EXAMPLES):
    temps = sorted(df["temperatura_c"].unique())
    idxs = []
    for T in temps:
        cand = np.where(df["temperatura_c"].to_numpy(float) == T)[0]
        if len(cand): idxs.append(int(cand[0]))
    rest = [i for i in range(len(df)) if i not in idxs]
    idxs += rest[:max(0, n - len(idxs))]
    return idxs[:n]

def plot_examples(fhz_khz, y_ref, X_orig, Y_comp, df, n=PLOT_N_EXAMPLES):
    idxs = pick_examples(df, n)
    for i in idxs:
        plt.figure(figsize=(8,4.5))
        plt.p

print("\n🔹 Gerando gráfico de exemplo...")
fhz_khz = fhz / 1e3
idx_show = df_test.index[10]  # primeira amostra do teste
plt.figure(figsize=(9, 5))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C (sem falha)")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C (falha={df.loc[idx_show,'falha']})")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado {df.loc[idx_show,'temperatura_c']}°C")
plt.title(f"Compensação RF — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.spatial.distance import cosine
from math import acos, degrees

def calc_metrics(y_true, y_pred):
    """Calcula todas as métricas entre curvas"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0,1]
    # SAM (Spectral Angle Mapper)
    sam_rad = acos(np.clip(np.dot(y_true, y_pred) /
                           (np.linalg.norm(y_true) * np.linalg.norm(y_pred) + 1e-12), -1, 1))
    sam_deg = degrees(sam_rad)
    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)
    rmsd = np.sqrt(np.mean((y_true - y_pred - np.mean(y_true - y_pred))**2))
    ccdm = 1 - corr
    return dict(R2=r2, RMSE=rmse, MAE=mae, Corr=corr,
                SAM_deg=sam_deg, NRMSE=nrmse, RMSD=rmsd, CCDM=ccdm)

# ===== Exemplo de uso =====
# y_ref: referência 20°C (ex: y_ref do código principal)
# X_orig: curva original (ex: df.loc[idx_show, fcols])
# X_comp: curva compensada (ex: df_comp.loc[idx_show, fcols])

y_ref_vec = y_ref
X_orig = df.loc[idx_show, fcols].to_numpy(float)
X_comp = df_comp.loc[idx_show, fcols].to_numpy(float)

print("\n== MÉTRICAS ORIGINAL vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_orig))

print("\n== MÉTRICAS COMPENSADO vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_comp))
